# Fine-tuning Qwen2-VL-2B with QLoRA for medical bill understandingReplaces the hosted Gemini call in **MediData** with an open-weight VLM you own.**Runtime → Change runtime type → T4 GPU** before running anything.Three stages, each warm-starting from the last:| stage | data | what it buys ||---|---|---|| A | RVL-CDIP (16-class docs) | the task *shape* — page image in, class label out || B | CORD-v2 receipts | line-item extraction with a defensible sample size || C | your 50 distilled medical pages | domain adaptation |Stage C alone cannot teach a 2B model both a new output format and a new domainfrom 35 training pages. A and B exist so C only has to do the second one.

In [ ]:
!nvidia-smiimport torchprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0,0)print("compute capability", cap, "| bf16 supported:", cap[0] >= 8)if cap[0] < 8:    print("\nTuring or older -> fp16 only, SDPA attention, no FlashAttention-2.")    print("The config defaults already assume this. Do not set VLM_USE_BF16=true here.")

## 1. Install

In [ ]:
!pip -q install "transformers>=4.49.0" "accelerate>=0.34.0" "peft>=0.13.0" \    "bitsandbytes>=0.44.0" "datasets>=3.0.0" pymupdf pillow sentencepiece!apt-get -qq install poppler-utils > /dev/nullprint("done")

## 2. Get the repoEither clone it, or upload the `vlm/` directory plus `attachments.zip`.

In [ ]:
import osREPO = "/content/Medical-Bill-Extract"BRANCH = "main"if not os.path.exists(REPO):    !git clone --branch {BRANCH} https://github.com/meet9614/Medical-Bill-Extract.git {REPO}else:    !cd {REPO} && git pullos.chdir(REPO)# vlm/ must exist here. If this raises, the local work has not been pushed yet:#   git add vlm app/local_backend.py requirements-vlm.txt tests/#   git commit -m "Add LoRA VLM track" && git pushassert os.path.isdir("vlm"), "vlm/ missing from the clone -- push it from your Mac first"!ls vlm

### Persist artifacts to Drive (do this before training)

In [ ]:
# Colab's disk is wiped when the session ends, and the free tier WILL# disconnect between stages. Without this, a lost session means retraining# stage A from scratch. Pointing artifacts at Drive makes each adapter survive,# so `--init stage_a` still resolves after a reconnect.from google.colab import drivedrive.mount('/content/drive')import osos.environ['VLM_ARTIFACT_ROOT'] = '/content/drive/MyDrive/medidata_vlm'os.makedirs(os.environ['VLM_ARTIFACT_ROOT'], exist_ok=True)print("artifacts ->", os.environ['VLM_ARTIFACT_ROOT'])# Datasets are large and re-downloadable; keep them on local disk, not Drive.os.environ['HF_HOME'] = '/content/hf_cache' 

## 3. Stage A — document classification on RVL-CDIP400 images/class = 6,400 examples. One epoch at effective batch 8 is ~800 steps,roughly 45-70 min on a T4. Drop `--per-class` to 150 for a quick smoke run.

In [ ]:
!python -m vlm.data.build_datasets --stage rvl_cdip --per-class 400

In [ ]:
!python -m vlm.train.train_lora --data stage_a_rvlcdip_train --out stage_a --epochs 1 --lr 1e-4

## 4. Stage B — line-item extraction on CORD-v2~800 receipts with real item/price/quantity annotations. Longer targets thanstage A, so expect ~2-3x the per-step time.

In [ ]:
!python -m vlm.data.build_datasets --stage cord!python -m vlm.train.train_lora --data stage_b_cord_train --out stage_b --init stage_a --epochs 2 --lr 1e-4

## 5. Stage C — adapt to your medical pagesRequires `GOOGLE_API_KEY` for the teacher pass. Run the distillation once; thelabels are cached.**The held-out split is not usable until you hand-correct it.** `distill.py`writes `test_review.csv` and the benchmark refuses to run against unreviewedteacher output — otherwise you would be measuring agreement with Gemini andcalling it accuracy.

In [ ]:
import os, getpassos.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")

In [ ]:
!python -m vlm.data.render --zip attachments.zip!python -m vlm.data.distill --dry-run

In [ ]:
!python -m vlm.data.distill

### Correct the held-out labels nowDownload `vlm/artifacts/labels/test_review.csv`, fix `corrected_page_type` byeye (it is ~15 rows), re-upload, then run the next cell.

In [ ]:
import json, pathlibp = pathlib.Path("vlm/artifacts/labels/VERIFIED")p.write_text(json.dumps({"verified": True, "note": "hand-reviewed"}))print("verified")

In [ ]:
!python -m vlm.data.build_datasets --stage medical!python -m vlm.train.train_lora --data stage_c_medical_train --out stage_c --init stage_b --epochs 6 --lr 5e-5

## 6. BenchmarkRuns both backends over the held-out pages and writes a JSON + markdown reportwith bootstrap CIs on every accuracy number and a breakeven volume on cost.`--training-gpu-hours` should be the *total* wall time across stages A+B+C, sothe amortised cost is real.

In [ ]:
!python -m vlm.eval.benchmark --backends local gemini --adapter stage_c \    --deployment T4 --training-gpu-hours 3.0

### Ablation: does the fine-tune actually help?Same harness against the **unadapted** base model. If stage_c is not clearlyabove this, the fine-tune did nothing and the honest report says so.

In [ ]:
!python -m vlm.eval.benchmark --backends local --adapter none \    --task classify

## 7. Save the adapterThe adapter is ~40MB — that is the entire artifact. The base weights are public.

In [ ]:
!cd vlm/artifacts/adapters && tar czf /content/stage_c_adapter.tar.gz stage_cfrom google.colab import filesfiles.download("/content/stage_c_adapter.tar.gz")

## What to do with the numbersRead `vlm/artifacts/results/benchmark_*.md`. Two rules before quoting anything:1. If the paired CI straddles zero, you have not measured a gap. Say   "indistinguishable at n=15", not a point estimate.2. Quote cost as a breakeven volume, not a multiplier. "Cheaper above ~N pages   on a T4; the API wins below that" is a claim that survives follow-up   questions. "4x cheaper" is not, because it hides the volume assumption.